# Yearly Wasserstein Distance Anomaly Analysis

**Part of:** Ethereum Topological Anomaly Detection (ETH-TAD)  
**Paper:** Ofori-Boateng et al. (2021) - arXiv:2106.01806

## Description

Statistical analysis and multi-method anomaly detection on Wasserstein distance time series.

**Methods:**
1. **S-ESD** - Seasonal Extreme Studentised Deviate
2. **IQR** - Interquartile Range fence
3. **Z-Score** - Rolling window z-scores
4. **Isolation Forest** - Unsupervised outlier detection
5. **LOF** - Local Outlier Factor
6. **Ensemble** - Consensus voting (≥3 methods)

## Prerequisites
- Completed notebook 3 (TDA analysis)
- Required data: `run_results_V8_{YEAR}.json` files

## Outputs
- Individual anomaly reports per year/layer
- Comparison summary across multiple runs

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(''), 'functions'))

import warnings
warnings.filterwarnings('ignore')

from yearly_analysis_functions import (
    load_runs,
    run_full_analysis,
    plot_comparison,
)

print('Imports OK ✓')

## Configuration

**Edit only this cell between runs.**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# YEARS & LAYERS TO ANALYZE
# ══════════════════════════════════════════════════════════════════════════

YEARS = [2020,2021,2022,2023,2024,2025]  # Add more years: [2020, 2021, 2022]

# Which layers to analyze (from TDA run results)
LAYERS = [
    # 'contract_txs_ETH_only_750',
    # 'simple_txs_ETH_only_750',
    'contract_txs_ALL_750',
    'simple_txs_ALL_750',
]

RESULTS_PREFIX = 'run_results_V1'   # files: {RESULTS_PREFIX}_{year}.json

# ══════════════════════════════════════════════════════════════════════════
# ANALYSIS PARAMETERS
# ══════════════════════════════════════════════════════════════════════════

ANALYSIS_PARAMS = {
    'sesd_period'      : 7,      # Seasonal period (days)
    'sesd_alpha'       : 0.05,   # Significance level
    'iqr_k'            : 1.5,    # IQR multiplier (1.5=standard, 3.0=extreme)
    'zscore_threshold' : 3.0,    # Z-score cutoff
    'zscore_window'    : 30,     # Rolling window (days)
    'contamination'    : 0.05,   # Expected anomaly fraction (5%)
    'lof_neighbors'    : 20,     # LOF n_neighbors
    'ensemble_votes'   : 3,      # Min methods to agree for ensemble
}

OUTPUT_DIR = './anomaly_reports'

print(f'Years: {YEARS}')
print(f'Layers: {LAYERS}')
print(f'Output: {OUTPUT_DIR}')
print('\nConfiguration set ✓')

## Load Data

In [ ]:
run_data = load_runs(YEARS, LAYERS, RESULTS_PREFIX)

## Anomaly Detection — All Runs

Runs each (year, layer) combination independently and saves an individual report per run.

In [ ]:
all_results = {}

for (year, layer), series in run_data.items():
    print(f'\n{"="*70}')
    print(f'  {year}  ·  {layer}')
    print(f'{"="*70}')

    all_results[(year, layer)] = run_full_analysis(
        series,
        title_suffix=f'  |  {year}  ·  {layer}',
        output_path=f'{OUTPUT_DIR}/anomaly_report_{year}_{layer}.png',
        **ANALYSIS_PARAMS
    )

## Comparison Across All Runs

In [ ]:
comparison_df = plot_comparison(
    run_data,
    all_results,
    output_dir=OUTPUT_DIR,
)

## Inspect & Export

In [ ]:
print('\nStatistical Comparison:')
display(comparison_df[[
    'label', 'N', 'Mean', 'Std Dev', 'Skewness', 'CV (%)',
    'Ensemble_count', 'S-ESD_count', 'IQR_count'
]])

print('\nAnomaly Method Comparison:')
display(comparison_df[[
    'label', 'S-ESD_count', 'IQR_count', 'Z-Score_count',
    'Isolation Forest_count', 'LOF_count', 'Ensemble_count'
]])

In [ ]:
import json

anomaly_export = {}

for (year, layer), result in all_results.items():
    label_str = f'{year}_{layer}'
    series = run_data[(year, layer)]
    ensemble_dates = series[result['ensemble']].index.strftime('%Y-%m-%d').tolist()

    anomaly_export[label_str] = {
        'ensemble_dates': ensemble_dates,
        'ensemble_count': len(ensemble_dates),
        'method_counts' : result['method_counts'],
    }

export_path = f'{OUTPUT_DIR}/anomaly_dates.json'
with open(export_path, 'w') as f:
    json.dump(anomaly_export, f, indent=2)

print(f'[\u2713] Anomaly dates exported \u2192 {export_path}')